In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# ==========================================
# BƯỚC 1: ĐỌC DỮ LIỆU TỪ FILE CSV KAGGLE
# ==========================================
print("1. Đang đọc dữ liệu từ file train.csv...")
# Đọc file CSV bằng Pandas
# FIX: Added on_bad_lines='skip' to handle potential malformed lines in train.csv.
# This will skip lines that have an unexpected number of fields, allowing the rest of the file to be read.
# Be aware that skipping lines might lead to data loss.
dataset = pd.read_csv('train.csv', on_bad_lines='skip')

# FIX: Check for and drop rows with NaN values after initial load
if dataset.isnull().values.any():
    print("Warning: NaN values detected in the raw dataset after loading. Attempting to drop rows with NaNs.")
    initial_rows = dataset.shape[0]
    dataset.dropna(inplace=True)
    rows_after_drop = dataset.shape[0]
    print(f"Dropped {initial_rows - rows_after_drop} rows containing NaN values.")
    if dataset.empty:
        raise ValueError("Dataset became empty after dropping rows with NaNs. Cannot proceed.")

# Tách nhãn (Cột 'label') và đặc trưng pixel (Các cột còn lại)
y_full = dataset['label'].values
X_full = dataset.drop('label', axis=1).values

# Reshape mảng 1D (784) về ảnh 2D (28x28x1) và chuẩn hóa / 255.0
X_full = X_full.reshape(-1, 28, 28, 1) / 255.0

# Mã hóa One-hot cho nhãn
y_full = tf.keras.utils.to_categorical(y_full, 10)

# Chia dữ liệu theo tỷ lệ 70% Train - 15% Validation - 15% Test
# Cắt 30% ra làm phần dư (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(X_full, y_full, test_size=0.3, random_state=42)
# Chia đôi phần dư để được 15% Val và 15% Test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"- Số lượng mẫu huấn luyện (Train): {X_train.shape[0]}")
print(f"- Số lượng mẫu xác thực (Val): {X_val.shape[0]}")
print(f"- Số lượng mẫu kiểm tra (Test): {X_test.shape[0]}")

# ==========================================
# BƯỚC 2: XÂY DỰNG MÔ HÌNH CNN (Giữ nguyên)
# ==========================================
print("\n2. Đang khởi tạo mô hình CNN...")
# FIX: Add Input layer and BatchNormalization for stability
model = models.Sequential([
    tf.keras.Input(shape=(28, 28, 1)), # Recommended way to specify input shape
    layers.Conv2D(32, (3, 3)),
    layers.BatchNormalization(), # Add Batch Normalization
    layers.ReLU(), # Explicit ReLU activation
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3)),
    layers.BatchNormalization(), # Add Batch Normalization
    layers.ReLU(), # Explicit ReLU activation
    layers.MaxPooling2D((2, 2)),

    layers.Dropout(0.25),
    layers.Flatten(),

    layers.Dense(128),
    layers.BatchNormalization(), # Add Batch Normalization
    layers.ReLU(), # Explicit ReLU activation
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# ==========================================
# BƯỚC 3: HUẤN LUYỆN MÔ HÌNH
# ==========================================
print("3. Bắt đầu huấn luyện...")
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(X_train, y_train,
                    epochs=15,
                    batch_size=64,
                    validation_data=(X_val, y_val),
                    callbacks=[early_stop])

# ==========================================
# BƯỚC 4: LẤY SỐ LIỆU CHO BÁO CÁO (CHƯƠNG 4)
# ==========================================
print("\n==========================================")
print("KẾT QUẢ ĐÁNH GIÁ ĐỂ ĐIỀN VÀO BẢNG 4.4.1")
print("==========================================")
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print(f"- Tập Train      | Loss: {train_loss:.4f} | Accuracy: {train_acc*100:.2f}%")

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"- Tập Validation | Loss: {val_loss:.4f} | Accuracy: {val_acc*100:.2f}%")

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"- Tập Test       | Loss: {test_loss:.4f} | Accuracy: {test_acc*100:.2f}%")
print("==========================================\n")

model.save('digit_model_kaggle.h5')

# ==========================================
# BƯỚC 5: VẼ BIỂU ĐỒ (Giữ nguyên)
# ==========================================
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='blue', marker='o')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange', marker='o')
plt.title('Đường cong độ chính xác (Accuracy)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='blue', marker='o')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange', marker='o')
plt.title('Đường cong độ mất mát (Loss)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print("Đang vẽ Ma trận nhầm lẫn...")
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true_classes, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Ma trận nhầm lẫn (Confusion Matrix) trên tập Test tự chia')
plt.xlabel('Nhãn Mô hình Dự đoán')
plt.ylabel('Nhãn Thực tế')
plt.show()

# New Section

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

print("1. Đang đọc dữ liệu từ file train.csv...")
dataset = pd.read_csv('train.csv')

# Tiền xử lý
y_full = tf.keras.utils.to_categorical(dataset['label'].values, 10)
X_full = dataset.drop('label', axis=1).values.reshape(-1, 28, 28, 1) / 255.0

# Chia tập dữ liệu
X_train, X_temp, y_train, y_temp = train_test_split(X_full, y_full, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("2. Đang khởi tạo và huấn luyện mô hình CNN (Vui lòng đợi khoảng 2-3 phút)...")
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Bắt đầu huấn luyện
model.fit(X_train, y_train, epochs=15, batch_size=64, validation_data=(X_val, y_val), callbacks=[early_stop])

# Lưu lại mô hình
model.save('digit_model_kaggle.h5')
print("\n✅ ĐÃ TẠO VÀ LƯU THÀNH CÔNG FILE 'digit_model_kaggle.h5'!")

1. Đang đọc dữ liệu từ file train.csv...
2. Đang khởi tạo và huấn luyện mô hình CNN (Vui lòng đợi khoảng 2-3 phút)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 36s 74ms/step - accuracy: 0.9211 - loss: 0.2601 - val_accuracy: 0.9727 - val_loss: 0.0866
Epoch 2/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 34s 60ms/step - accuracy: 0.9768 - loss: 0.0746 - val_accuracy: 0.9776 - val_loss: 0.0682
Epoch 3/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 41s 60ms/step - accuracy: 0.9839 - loss: 0.0521 - val_accuracy: 0.9868 - val_loss: 0.0441
Epoch 4/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 28s 60ms/step - accuracy: 0.9868 - loss: 0.0405 - val_accuracy: 0.9871 - val_loss: 0.0422
Epoch 5/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 28s 61ms/step - accuracy: 0.9883 - loss: 0.0354 - val_accuracy: 0.9879 - val_loss: 0.0384
Epoch 6/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 40s 60ms/step - accuracy: 0.9913 - loss: 0.0269 - val_accuracy: 0.9856 - val_loss: 0.0432
Epoch 7/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 41s 59ms/step - accuracy: 0.9916 - loss: 0.0256 - val_accuracy: 0.9873 - val_loss: 0.0401
Epoch 8/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 27s 59ms/step - accuracy: 0.9938 - loss: 0.0199 - 


✅ ĐÃ TẠO VÀ LƯU THÀNH CÔNG FILE 'digit_model_kaggle.h5'!


In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
import cv2

# Tải lại mô hình
model = tf.keras.models.load_model('digit_model_kaggle.h5')

def predict_digit(img):
    if img is None:
        return "Vui lòng vẽ một chữ số!"

    # 1. Trích xuất mảng ảnh từ Gradio Sketchpad
    if isinstance(img, dict):
        img_array = img['composite']
    else:
        img_array = img

    # Chuyển sang ảnh xám (Grayscale)
    if len(img_array.shape) == 3:
        img_gray = cv2.cvtColor(img_array, cv2.COLOR_BGR2GRAY)
    else:
        img_gray = img_array

    # 2. CHUYỂN ĐỔI MÀU: Nền trắng -> đen, Nét đen -> trắng
    # Sử dụng Threshold để khử nhiễu hoàn toàn, lấy nét trắng tinh (255)
    _, img_thresh = cv2.threshold(img_gray, 128, 255, cv2.THRESH_BINARY_INV)

    # 3. LÀM ĐẬM NÉT VẼ (Dilation)
    # Vì nét vẽ chuột thường mỏng, ta dùng kernel để làm nét chữ béo lên giống bút dạ
    kernel = np.ones((15, 15), np.uint8)
    img_dilated = cv2.dilate(img_thresh, kernel, iterations=1)

    # 4. CẮT SÁT VIỀN VÀ CĂN GIỮA (Bounding box chuẩn MNIST)
    coords = cv2.findNonZero(img_dilated)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        digit = img_dilated[y:y+h, x:x+w]

        # Ép tỷ lệ: Chiều dài nhất sẽ bằng 20 pixel (chuẩn MNIST)
        if w > h:
            new_w = 20
            new_h = int(20 * (h/w))
        else:
            new_h = 20
            new_w = int(20 * (w/h))

        digit_resized = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)

        # Thêm viền đen xung quanh để tạo thành ảnh 28x28 hoàn hảo
        pad_top = (28 - new_h) // 2
        pad_bottom = 28 - new_h - pad_top
        pad_left = (28 - new_w) // 2
        pad_right = 28 - new_w - pad_left

        img_ready = cv2.copyMakeBorder(digit_resized, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_CONSTANT, value=0)
    else:
        img_ready = cv2.resize(img_dilated, (28, 28))

    # 5. Đưa vào mô hình dự đoán
    img_normalized = img_ready / 255.0
    img_reshaped = img_normalized.reshape(1, 28, 28, 1)

    prediction = model.predict(img_reshaped)
    predicted_class = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    return f"Mô hình dự đoán: Số {predicted_class} (Độ tin cậy: {confidence:.2f}%)"

# Tạo giao diện
interface = gr.Interface(
    fn=predict_digit,
    inputs=gr.Sketchpad(label="Khung vẽ (Nên vẽ số ở giữa)"),
    outputs=gr.Label(label="Kết quả"),
    title="PHẦN MỀM NHẬN DẠNG CHỮ SỐ VIẾT TAY (BẢN CHUẨN)",
    description="Hệ thống đã tích hợp OpenCV để tự động làm đậm nét vẽ và căn giữa theo chuẩn dữ liệu MNIST."
)

interface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4f752d642bfbcbca9a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4f752d642bfbcbca9a.gradio.li

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

print("1. Đang đọc dữ liệu từ file train.csv...")
dataset = pd.read_csv('train.csv')

# Tiền xử lý
y_full = tf.keras.utils.to_categorical(dataset['label'].values, 10)
X_full = dataset.drop('label', axis=1).values.reshape(-1, 28, 28, 1) / 255.0

# Chia tập dữ liệu
X_train, X_temp, y_train, y_temp = train_test_split(X_full, y_full, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("2. Đang khởi tạo và huấn luyện mô hình CNN (Vui lòng đợi khoảng 2-3 phút)...")
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Bắt đầu huấn luyện
model.fit(X_train, y_train, epochs=15, batch_size=64, validation_data=(X_val, y_val), callbacks=[early_stop])

# Lưu lại mô hình
model.save('digit_model_kaggle.h5')
print("\n✅ ĐÃ TẠO VÀ LƯU THÀNH CÔNG FILE 'digit_model_kaggle.h5'!")

1. Đang đọc dữ liệu từ file train.csv...
2. Đang khởi tạo và huấn luyện mô hình CNN (Vui lòng đợi khoảng 2-3 phút)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 30s 61ms/step - accuracy: 0.9189 - loss: 0.2644 - val_accuracy: 0.9719 - val_loss: 0.0863
Epoch 2/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 41s 61ms/step - accuracy: 0.9751 - loss: 0.0809 - val_accuracy: 0.9787 - val_loss: 0.0632
Epoch 3/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 40s 60ms/step - accuracy: 0.9815 - loss: 0.0576 - val_accuracy: 0.9835 - val_loss: 0.0505
Epoch 4/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 28s 61ms/step - accuracy: 0.9861 - loss: 0.0444 - val_accuracy: 0.9832 - val_loss: 0.0514
Epoch 5/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 41s 61ms/step - accuracy: 0.9883 - loss: 0.0363 - val_accuracy: 0.9873 - val_loss: 0.0367
Epoch 6/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 41s 62ms/step - accuracy: 0.9906 - loss: 0.0288 - val_accuracy: 0.9887 - val_loss: 0.0349
Epoch 7/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 27s 59ms/step - accuracy: 0.9911 - loss: 0.0259 - val_accuracy: 0.9871 - val_loss: 0.0426
Epoch 8/15
460/460 ━━━━━━━━━━━━━━━━━━━━ 27s 59ms/step - accuracy: 0.9930 - loss: 0.0215 - 


✅ ĐÃ TẠO VÀ LƯU THÀNH CÔNG FILE 'digit_model_kaggle.h5'!
